<div
  style="
    background-color: #f0f0f0;
    color:rgb(56, 56, 56);
    padding: 8px;
    display: flex;
    align-items: center;
    gap: 100px;
  "
>
  <img src="./images/brand.svg" style="max-height: 80px;">
  <strong>
    AI Saga: Deep Learning & Generative AI<br/>
    Lab 6: Chain-of-Thought to Agent — A Progressive Journey<br/>
  </strong>
  <emph>
    Student Name: [Complete with your name]<br/>
    Date: [Fill up with the submission date]<br/>
  </emph>
</div>

### Background

This lab takes you on a progressive journey from basic prompting to building a working AI agent. You'll experience firsthand how reasoning techniques like Chain-of-Thought (CoT) improve LLM performance, implement a simplified Tree-of-Thoughts (ToT) algorithm, and finally create a ReAct agent capable of using tools to solve real problems.

This lab directly connects to your capstone project: the research assistant you'll build uses the same patterns you'll implement here.

---

### Description

Check LMS for the last updated instructions.

You will work through three main parts:
1. **Part 1: Warm-up — Direct vs Chain-of-Thought** - Compare direct prompting with CoT reasoning
2. **Part 2: Core — Simplified Tree-of-Thoughts** - Implement branching reasoning with evaluation
3. **Part 3: Capstone — ReAct Agent with Tools** - Build a fact-checking agent using the ReAct pattern

---

### Deliverables

- A completed **6.lab.1.reasoning_to_agent.ipynb** Jupyter notebook containing:
  * All code for Parts 1-3
  * Comparison tables and trace visualizations
  * Working implementations with test outputs
- Brief reflection discussing trade-offs between reasoning strategies

#### Requirements

- Use **OpenAI API**, **Anthropic API**, or a local model
- Code must be well-commented and clearly organized
- Include markdown cells explaining each step and your design decisions

---

### Key Reminders

**Think scientifically** - design experiments with clear hypotheses and systematic methodology.

**Analyze trade-offs** - consider accuracy, cost (tokens), and latency for each approach.

**Note**: All submitted code is code that you must be able to defend (even if AI generated!).

In [ ]:
# Important Notice
#
# Please make sure to check the official instructions for this assignment in Canvas LMS
# as they may have been updated or changed. The instructions above are provided for
# reference only and may not reflect the most current requirements.
#
# Always refer to Canvas LMS for:
# - Latest assignment requirements
# - Due dates
# - Grading criteria
# - Any special instructions
#
# When in doubt, ask your instructor for clarification.

# Install dependencies (uncomment if needed)
!pip install openai anthropic python-dotenv

import os
import json
import time
import re
from typing import List, Dict, Callable, Any, Optional, Tuple
from dataclasses import dataclass, field
from enum import Enum

import pandas as pd
import matplotlib.pyplot as plt

# Choose your API provider (uncomment one)
from openai import OpenAI

# from anthropic import Anthropic

import warnings

warnings.filterwarnings("ignore")

# Load API key from environment or .env file
from dotenv import load_dotenv

load_dotenv()

print("Setup complete!")

---

## Setup: API Configuration

Configure your API client. You can use OpenAI, Anthropic, or a local model.

In [ ]:
# TODO: Configure your API client
# Option 1: OpenAI
client = OpenAI(api_key="")
MODEL = "gpt-4o-mini"  # or "gpt-4o" for better results

# Option 2: Anthropic
# client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
# MODEL = "claude-3-haiku-20240307"  # or "claude-sonnet-4-20250514"


# Helper function to call LLM (adapt based on your provider)
def call_llm(
    prompt: str, temperature: float = 0.7, max_tokens: int = 500
) -> Tuple[str, int]:
    """
    Call the LLM and return (response_text, token_count).

    Args:
        prompt: The input prompt
        temperature: Controls randomness (0.0 = deterministic, 1.0 = creative)
        max_tokens: Maximum tokens in response

    Returns:
        tuple: (response_text, total_tokens_used)
    """
    # TODO: Implement based on your chosen provider

    # Example for OpenAI:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content, response.usage.total_tokens

    # Example for Anthropic:
    # response = client.messages.create(
    #     model=MODEL,
    #     max_tokens=max_tokens,
    #     messages=[{"role": "user", "content": prompt}]
    # )
    # return response.content[0].text, response.usage.input_tokens + response.usage.output_tokens

    raise NotImplementedError("Configure your API client above!")


# Test the connection
response, tokens = call_llm("Say 'Hello, World!'")
print(f"Response: {response}")
print(f"Tokens used: {tokens}")

---

## Part 1: Warm-up — Direct vs Chain-of-Thought

**Goal:** Experience the difference between direct prompting and Chain-of-Thought reasoning.

### Tasks:

1. Solve 3 problems using **direct prompting** (just ask the question)
2. Solve the same 3 problems using **CoT prompting** ("Let's think step by step...")
3. Compare: accuracy, reasoning quality, and token count

### Step 1.1: Define Test Problems

In [ ]:
# Test problems with known answers
test_problems = [
    {
        "id": 1,
        "type": "math",
        "question": "A store sells apples for $2 each. If I buy 3 apples and pay with $10, how much change do I get?",
        "expected_answer": "$4",
    },
    {
        "id": 2,
        "type": "logic",
        "question": "If all roses are flowers and some flowers fade quickly, can we conclude that some roses fade quickly?",
        "expected_answer": "No",  # This is a logical fallacy (undistributed middle)
    },
    {
        "id": 3,
        "type": "multi-step",
        "question": "I have 24 cookies. I give half to my sister, then eat 4. How many are left?",
        "expected_answer": "8",
    },
]

print(f"Loaded {len(test_problems)} test problems")
for p in test_problems:
    print(f"  [{p['type']}] {p['question'][:50]}...")

### Step 1.2: Direct Prompting

In [ ]:
def direct_prompt(question: str) -> str:
    """Create a direct prompt (no reasoning instructions)."""
    return f"{question}\n\nAnswer:"


# Store results
direct_results = []

# TODO: Run direct prompting for each problem
for problem in test_problems:
    prompt = direct_prompt(problem["question"])

    print(f"\n{'='*80}")
    print(f"Problem {problem['id']} ({problem['type']}): {problem['question']}")
    print(f"{'='*80}")
    print(f"\nPrompt:\n{prompt}")

    # TODO: Uncomment after configuring API
    response, tokens = call_llm(
        prompt, temperature=0.0
    )  # temperature=0 for consistency
    print(f"\nResponse:\n{response}")
    print(f"\nTokens used: {tokens}")
    print(f"Expected answer: {problem['expected_answer']}")

    user_eval = input("Was it correct? (y/n): ").strip().lower()
    is_correct = user_eval == "y"

    direct_results.append(
        {
            "problem_id": problem["id"],
            "type": problem["type"],
            "method": "direct",
            "response": response,
            "tokens": tokens,
            "expected": problem["expected_answer"],
            "correct": is_correct,
        }
    )

### Step 1.3: Chain-of-Thought Prompting

In [ ]:
def cot_prompt(question: str) -> str:
    """Create a Chain-of-Thought prompt."""
    return f"{question}\n\nLet's think step by step:"


# Store results
cot_results = []

# TODO: Run CoT prompting for each problem
for problem in test_problems:
    prompt = cot_prompt(problem["question"])

    print(f"\n{'='*80}")
    print(f"Problem {problem['id']} ({problem['type']}): {problem['question']}")
    print(f"{'='*80}")
    print(f"\nPrompt:\n{prompt}")

    # TODO: Uncomment after configuring API
    response, tokens = call_llm(prompt, temperature=0.0)
    print(f"\nResponse:\n{response}")
    print(f"\nTokens used: {tokens}")
    print(f"Expected answer: {problem['expected_answer']}")

    user_eval = input("Was it correct? (y/n): ").strip().lower()
    is_correct = user_eval == "y"

    cot_results.append(
        {
            "problem_id": problem["id"],
            "type": problem["type"],
            "method": "cot",
            "response": response,
            "tokens": tokens,
            "expected": problem["expected_answer"],
            "correct": is_correct,  # TODO: Fill manually after reviewing
        }
    )

### Step 1.4: Comparison Analysis

In [ ]:
# TODO: Create comparison DataFrame after running experiments
all_results = direct_results + cot_results
results_df = pd.DataFrame(all_results)

# Example comparison table structure
comparison_table = pd.DataFrame(
    {
        "Problem": ["Math", "Logic", "Multi-step"],
        "Direct_Correct": ["True", "True", "True"],  # TODO: Fill with True/False
        "Direct_Tokens": [96, 104, 120],  # TODO: Fill with token counts
        "CoT_Correct": ["True", "True", "True"],  # TODO: Fill with True/False
        "CoT_Tokens": [202, 298, 194],  # TODO: Fill with token counts
    }
)

print("Comparison Table (fill after running experiments):")
print(comparison_table.to_string(index=False))

In [ ]:
# TODO: Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# # Accuracy comparison
accuracy_data = results_df.groupby("method")["correct"].mean()
accuracy_data.plot(kind="bar", ax=axes[0], color=["steelblue", "green"])
axes[0].set_title("Accuracy: Direct vs CoT")
axes[0].set_ylabel("Accuracy")
axes[0].set_ylim(0, 1)

# # Token usage comparison
token_data = results_df.groupby("method")["tokens"].mean()
token_data.plot(kind="bar", ax=axes[1], color=["steelblue", "green"])
axes[1].set_title("Token Usage: Direct vs CoT")
axes[1].set_ylabel("Average Tokens")

plt.tight_layout()
plt.show()

### Part 1 Analysis

**1. Which method performed better on accuracy?**

Both methods showed a very similar level of accuracy. In my case, both the direct approach and the Chain-of-Thought responded correctly to most problems, probably influenced by the chosen model, which is already quite robust even without reasoning instructions.

**2. What was the token cost difference?**

The Chain-of-Thought method consumed more tokens in all tests. This was expected, as the model generates step-by-step explanations and therefore requires more text to arrive at the same answer.

**3. For which problem types did CoT help the most?**

CoT would be particularly useful in problems requiring multiple steps of reasoning, intermediate calculations, or logical analysis. Although both methods were accurate in this experiment, CoT provides greater robustness in tasks involving logic or problem decomposition.

**4. When might direct prompting be preferable?**

Direct prompting is more suitable for simple tasks or questions that do not require much reasoning. It is faster, cheaper in tokens, and sufficient when the model can give the answer without needing to explain its internal process.

---

## Part 2: Core — Simplified Tree-of-Thoughts

**Goal:** Implement branching reasoning with self-evaluation.

### The "24 Game"

Given 4 numbers, use +, -, x, / to make 24.

Example: [4, 8, 3, 1] -> (8 / 4) x (3 x 1) = ... Try again!

### Step 2.1: Define ToT Components

In [ ]:
class ThoughtScore(Enum):
    """Evaluation scores for thoughts."""

    PROMISING = "promising"
    NEUTRAL = "neutral"
    DEAD_END = "dead_end"


@dataclass
class Thought:
    """A single reasoning step."""

    content: str
    remaining_numbers: List[float]
    intermediate_result: Optional[float]
    score: Optional[ThoughtScore] = None
    children: List["Thought"] = field(default_factory=list)


def format_state(numbers: List[float], target: int = 24) -> str:
    """Format current state for the prompt."""
    return f"Numbers available: {numbers}\nTarget: {target}"

### Step 2.2: Implement Generate Function

In [ ]:
def parse_steps(response: str) -> List[Dict[str, any]]:
    """
    Parse LLM response into structured steps.

    Args:
        response: Raw LLM output string

    Returns:
        List of step dictionaries with keys: description, result, remaining
    """
    steps = []

    step_pattern = r"STEP:\s*(.*?)(?:\n|$)"
    result_pattern = r"RESULT:\s*([+-]?\d*\.?\d+)(?:\n|$)"
    remaining_pattern = r"REMAINING:\s*\[([^\]]+)\](?:\n|$)"

    sections = re.split(r"\n\s*\n", response.strip())

    for section in sections:
        if not section.strip():
            continue

        step_match = re.search(step_pattern, section)
        result_match = re.search(result_pattern, section)
        remaining_match = re.search(remaining_pattern, section)

        if step_match and result_match and remaining_match:
            description = step_match.group(1).strip()
            result = float(result_match.group(1))

            remaining_str = remaining_match.group(1).strip()
            remaining = [float(x.strip()) for x in remaining_str.split(",")]

            steps.append(
                {"description": description, "result": result, "remaining": remaining}
            )

    return steps


def generate_thoughts(
    numbers: List[float], n: int = 3, target: int = 24
) -> List[Thought]:
    """
    Generate n possible next steps from current state.

    Args:
        numbers: Available numbers
        n: Number of thoughts to generate
        target: Target number (24)

    Returns:
        List of Thought objects
    """
    prompt = f"""You are solving the 24 Game.
{format_state(numbers, target)}

Propose {n} different FIRST STEPS. Each step should:
1. Pick exactly 2 numbers from the list
2. Apply one operation (+, -, *, /)
3. Show the result and remaining numbers

Format each step as:
STEP: [operation description]
RESULT: [number]
REMAINING: [list of remaining numbers including result]

Propose {n} different steps:"""

    # TODO: Call LLM and parse response
    response, _ = call_llm(prompt, temperature=0.7)

    # TODO: Parse response into Thought objects
    thoughts = []

    # Example parsing (you'll need to adapt based on actual LLM output):
    for step in parse_steps(response):
        thoughts.append(
            Thought(
                content=step["description"],
                remaining_numbers=step["remaining"],
                intermediate_result=step["result"],
            )
        )

    return thoughts


# Test
thoughts = generate_thoughts([4, 8, 3, 1])
for i, t in enumerate(thoughts, 1):
    print(f"Thought {i}: {t.content}")

### Step 2.3: Implement Evaluate Function

In [ ]:
def evaluate_thought(thought: Thought, target: int = 24) -> ThoughtScore:
    """
    Evaluate if a thought is promising, neutral, or a dead end.

    Args:
        thought: The thought to evaluate
        target: Target number (24)

    Returns:
        ThoughtScore enum value
    """
    prompt = f"""You are evaluating a step in the 24 Game.

Step taken: {thought.content}
Result: {thought.intermediate_result}
Remaining numbers: {thought.remaining_numbers}
Target: {target}

Evaluate this step:
- PROMISING: This step brings us closer to 24, good intermediate values
- NEUTRAL: This step is valid but unclear if it helps
- DEAD_END: This step makes it impossible or very hard to reach 24

Respond with exactly one word: PROMISING, NEUTRAL, or DEAD_END"""

    # TODO: Call LLM and parse response
    response, _ = call_llm(prompt, temperature=0.0)
    score_str = response.strip().upper()

    # TODO: Map response to ThoughtScore
    if "PROMISING" in score_str:
        return ThoughtScore.PROMISING
    elif "DEAD_END" in score_str:
        return ThoughtScore.DEAD_END
    else:
        return ThoughtScore.NEUTRAL

    # return ThoughtScore.NEUTRAL  # Placeholder

### Step 2.4: Implement ToT Search

In [ ]:
def tot_solve(
    numbers: List[float], target: int = 24, max_depth: int = 4
) -> Optional[List[str]]:
    """
    Solve the 24 Game using Tree-of-Thoughts.

    Args:
        numbers: Starting numbers (4 numbers)
        target: Target number (24)
        max_depth: Maximum search depth

    Returns:
        List of steps if solution found, None otherwise
    """
    print(f"\nSolving: {numbers} -> {target}")
    print("=" * 50)

    # Track the search
    trace = []

    def search(
        current_numbers: List[float], path: List[str], depth: int
    ) -> Optional[List[str]]:
        """Recursive search with backtracking."""

        # Base case: only one number left
        if len(current_numbers) == 1:
            if abs(current_numbers[0] - target) < 0.001:
                print(f"  [OK] Solution found!")
                return path
            else:
                print(f"  [X] Dead end: {current_numbers[0]} != {target}")
                return None

        # Max depth reached
        if depth >= max_depth:
            print(f"  [X] Max depth reached")
            return None

        # Generate thoughts
        print(f"\n  Depth {depth}: Generating thoughts for {current_numbers}")
        thoughts = generate_thoughts(current_numbers, n=3, target=target)

        # Evaluate and sort by score
        for thought in thoughts:
            thought.score = evaluate_thought(thought, target)
            print(f"    -> {thought.content} [{thought.score.value}]")

        # Sort: promising first, then neutral, skip dead ends
        thoughts.sort(
            key=lambda t: (
                0
                if t.score == ThoughtScore.PROMISING
                else 1 if t.score == ThoughtScore.NEUTRAL else 2
            )
        )

        # Explore promising and neutral paths
        for thought in thoughts:
            if thought.score == ThoughtScore.DEAD_END:
                continue

            # Recurse
            result = search(
                thought.remaining_numbers, path + [thought.content], depth + 1
            )

            if result is not None:
                return result

            # Backtrack
            print(f"  <- Backtracking from: {thought.content}")

        return None

    # Start search
    solution = search(numbers, [], 0)

    if solution:
        print(f"\n[OK] SOLUTION FOUND:")
        for i, step in enumerate(solution, 1):
            print(f"  {i}. {step}")
    else:
        print(f"\n[X] No solution found within depth {max_depth}")

    return solution

### Step 2.5: Test ToT on Multiple Cases

In [ ]:
# Test cases for the 24 Game
test_cases = [
    [4, 8, 3, 1],  # (8 - 4) x (3 + 1) x ... or other solutions
    [2, 5, 6, 7],  # Multiple possible solutions
    [1, 2, 3, 4],  # 1 x 2 x 3 x 4 = 24
]

tot_results = []

# TODO: Run ToT for each test case
for numbers in test_cases:
    print(f"\n{'='*80}")
    print(f"TEST CASE: {numbers}")
    print(f"{'='*80}")

    # TODO: Uncomment after implementing
    start_time = time.time()
    solution = tot_solve(numbers)
    elapsed = time.time() - start_time

    tot_results.append(
        {
            "numbers": numbers,
            "solved": solution is not None,
            "steps": len(solution) if solution else 0,
            "time": elapsed,
        }
    )

# Summary
results_df = pd.DataFrame(tot_results)
print("\nToT Results Summary:")
print(results_df.to_string(index=False))

### Part 2 Analysis

**1. How did the evaluation function affect search efficiency?**

The evaluation function helped make the search a little more orderly, because it prioritized the steps that seemed closest to the solution. However, it also caused the algorithm to close too quickly on certain possibilities, which meant that it stopped exploring paths that could have led to a correct answer.

**2. When did backtracking occur? Was it effective?**

Backtracking occurred when the algorithm followed a path and encountered a dead end. In such cases, it would go back and try another path. In general, it was useful, but only within the scope of what the algorithm was able to explore.

**3. What are the limitations of this simplified ToT?**

The main limitation is that it explores very few possibilities at each step, so it is easy for it to overlook valid solutions. It also has a short search depth and relies heavily on an evaluation that does not always accurately reflect what might be a good path.

---

## Part 3: Capstone — ReAct Agent with Tools

**Goal:** Build a "Fact Checker" agent that verifies claims using tools.

### The ReAct Pattern

```
Thought: I need to verify this claim...
Action: search("Eiffel Tower height")
Observation: The Eiffel Tower is 330 meters tall...
Thought: Now I can compare...
Action: compare(330, 330)
Observation: Values are equal
Thought: The claim is verified!
Answer: TRUE
```

### Step 3.1: Define Tools

In [ ]:
# Tool implementations
def calculator(expression: str) -> str:
    """Evaluate a mathematical expression."""
    try:
        # Safe evaluation (only basic math)
        allowed_chars = set("0123456789+-*/.() ")
        if not all(c in allowed_chars for c in expression):
            return "Error: Invalid characters in expression"
        result = eval(expression)
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {str(e)}"


def search(query: str) -> str:
    """
    Mock search function with pre-defined results.
    In a real implementation, this would call a search API.
    """
    # Pre-defined search results for reproducibility
    knowledge_base = {
        "eiffel tower height": "The Eiffel Tower is 330 meters (1,083 ft) tall, including antennas. The structure itself is 300 meters.",
        "eiffel tower": "The Eiffel Tower is 330 meters (1,083 ft) tall, including antennas. The structure itself is 300 meters.",
        "python speed": "Python is an interpreted language, generally slower than compiled languages like C++. Python is typically 10-100x slower than C++ for CPU-intensive tasks.",
        "python vs c++": "Python is an interpreted language, generally slower than compiled languages like C++. Python is typically 10-100x slower than C++ for CPU-intensive tasks.",
        "python faster than c++": "This is false. Python is an interpreted language and is generally much slower than C++, a compiled language.",
        "alexander graham bell": "Alexander Graham Bell (1847-1922) was a Scottish-born inventor, scientist, and engineer who is credited with inventing the first practical telephone. He was born in Edinburgh, Scotland.",
        "telephone inventor": "Alexander Graham Bell (1847-1922) was a Scottish-born inventor who is credited with inventing the first practical telephone. He was born in Edinburgh, Scotland.",
        "inventor telephone birthplace": "Alexander Graham Bell, inventor of the telephone, was born in Edinburgh, Scotland on March 3, 1847.",
    }

    query_lower = query.lower()

    # Try exact match first
    if query_lower in knowledge_base:
        return knowledge_base[query_lower]

    # Try partial match
    for key, value in knowledge_base.items():
        if key in query_lower or query_lower in key:
            return value
        # Check for keyword overlap
        query_words = set(query_lower.split())
        key_words = set(key.split())
        if len(query_words & key_words) >= 2:
            return value

    return f"No results found for: {query}"


def compare(a: str, b: str) -> str:
    """Compare two values."""
    try:
        # Try numeric comparison
        num_a = float(a)
        num_b = float(b)
        if abs(num_a - num_b) < 0.001:
            return f"{a} equals {b}"
        elif num_a > num_b:
            return f"{a} is greater than {b}"
        else:
            return f"{a} is less than {b}"
    except ValueError:
        # String comparison
        if a.lower() == b.lower():
            return f"'{a}' equals '{b}'"
        else:
            return f"'{a}' does not equal '{b}'"


# Tool registry
TOOLS = {
    "calculator": calculator,
    "search": search,
    "compare": compare,
}

# Test tools
print("Testing tools:")
print(f"  calculator('2 + 2'): {calculator('2 + 2')}")
print(f"  search('eiffel tower height'): {search('eiffel tower height')[:50]}...")
print(f"  compare('330', '330'): {compare('330', '330')}")

### Step 3.2: Implement ReAct Agent

In [ ]:
def parse_action(response: str) -> Tuple[Optional[str], Optional[str]]:
    """
    Parse action from LLM response.

    Returns:
        tuple: (action_name, action_input) or (None, None) if no action
    """
    # Look for Action: tool_name(arguments)
    # Pattern: Action: tool_name("argument") or Action: tool_name(argument)
    pattern = r'Action:\s*(\w+)\s*\(["\']?(.+?)["\']?\)'
    match = re.search(pattern, response, re.IGNORECASE)

    if match:
        tool_name = match.group(1).lower()
        tool_input = match.group(2)
        return tool_name, tool_input

    return None, None


def execute_tool(action_name: str, action_input: str) -> str:
    """Execute a tool and return the observation."""
    if action_name not in TOOLS:
        return f"Error: Unknown tool '{action_name}'. Available tools: {list(TOOLS.keys())}"

    try:
        result = TOOLS[action_name](action_input)
        return result
    except Exception as e:
        return f"Error executing {action_name}: {str(e)}"

In [ ]:
def react_agent(claim: str, max_iterations: int = 5) -> Dict:
    """
    ReAct agent that verifies a claim using tools.

    Args:
        claim: The claim to verify
        max_iterations: Maximum number of thought-action-observation cycles

    Returns:
        dict with verdict, reasoning trace, and metadata
    """
    print(f"\n{'='*80}")
    print(f"CLAIM: {claim}")
    print(f"{'='*80}")

    # Build the system prompt
    system_prompt = f"""You are a fact-checking agent. Your task is to verify claims using available tools.

Available tools:
- search(query): Search for information about a topic
- calculator(expression): Evaluate a mathematical expression
- compare(a, b): Compare two values

Use the following format:

Thought: [your reasoning about what to do next]
Action: tool_name("argument")

After receiving an Observation, continue with another Thought/Action or provide your final answer:

Thought: [final reasoning]
Answer: TRUE, FALSE, or UNVERIFIABLE

Claim to verify: {claim}"""

    # Conversation history
    history = [system_prompt]
    trace = []  # Store (thought, action, observation) tuples

    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")

        # Build prompt from history
        prompt = "\n\n".join(history)

        # Get LLM response
        # TODO: Uncomment after configuring API
        response, tokens = call_llm(prompt, temperature=0.0)
        print(f"\nAgent:\n{response}")

        # Placeholder for testing
        # response = "Thought: I need to implement the API call\nAnswer: UNVERIFIABLE"
        # print(f"\nAgent:\n{response}")

        # Check if agent has reached a conclusion
        if "Answer:" in response:
            # Extract answer
            answer_line = [l for l in response.split("\n") if "Answer:" in l]
            if answer_line:
                verdict = answer_line[0].split("Answer:")[1].strip().upper()
                if verdict in ["TRUE", "FALSE", "UNVERIFIABLE"]:
                    print(f"\n[OK] VERDICT: {verdict}")
                    return {
                        "claim": claim,
                        "verdict": verdict,
                        "trace": trace,
                        "iterations": iteration + 1,
                    }

        # Parse and execute action
        action_name, action_input = parse_action(response)

        if action_name:
            print(f"\nExecuting: {action_name}({action_input})")
            observation = execute_tool(action_name, action_input)
            print(f"Observation: {observation}")

            # Add to history
            history.append(response)
            history.append(f"Observation: {observation}")

            # Record trace
            trace.append(
                {
                    "thought": response.split("Action:")[0].strip(),
                    "action": f"{action_name}({action_input})",
                    "observation": observation,
                }
            )
        else:
            # No action found, add response and prompt for action
            history.append(response)
            history.append("Please provide an Action or Answer.")

    print(f"\n[X] Max iterations reached without verdict")
    return {
        "claim": claim,
        "verdict": "UNVERIFIABLE",
        "trace": trace,
        "iterations": max_iterations,
    }

### Step 3.3: Test the Agent

In [ ]:
# Test claims
test_claims = [
    {"claim": "The Eiffel Tower is 330 meters tall.", "expected": "TRUE"},
    {"claim": "Python is faster than C++.", "expected": "FALSE"},
    {
        "claim": "The inventor of the telephone was born in Scotland.",
        "expected": "TRUE",  # Alexander Graham Bell was born in Edinburgh
    },
]

agent_results = []

# TODO: Run agent for each claim
for test in test_claims:
    print(f"\n{'#'*80}")
    print(f"TESTING: {test['claim']}")
    print(f"EXPECTED: {test['expected']}")
    print(f"{'#'*80}")

    # TODO: Uncomment after implementing
    result = react_agent(test["claim"])
    result["expected"] = test["expected"]
    result["correct"] = result["verdict"] == test["expected"]
    agent_results.append(result)

    # Placeholder
    # agent_results.append({
    #     "claim": test["claim"],
    #     "verdict": "TODO",
    #     "expected": test["expected"],
    #     "correct": None
    # })

In [ ]:
# Summary
summary_df = pd.DataFrame(
    [
        {
            "Claim": r["claim"][:40] + "...",
            "Expected": r["expected"],
            "Verdict": r["verdict"],
            "Correct": r.get("correct", "N/A"),
        }
        for r in agent_results
    ]
)

print("\nAgent Results Summary:")
print(summary_df.to_string(index=False))

### Part 3 Analysis

**1. How did the agent use tools to verify claims?**

What I noticed is that the agent decides which tool to use based on what the prompt says. Basically, the prompt explains what tools are available and what they are used for, and with that information, the model analyzes each claim and chooses the tool that best fits to verify it. In other words, it doesn't "guess, it simply follows the options we give it and chooses the one that makes the most sense for each case.

**2. What errors or inefficiencies did you observe?**

Actually, the agent performed quite well. The claims I tested were not that complex, and the tools already returned clear data, so the agent only had to review that information and give its final verdict. The only thing I noticed is that if the number of iterations is greatly reduced, it may not have a chance to complete the analysis and reach a conclusion, especially if the process requires several steps.

**3. How would you improve the agent?**

One improvement would be to stop relying on hardcoded data in the code. Instead, the ideal solution would be to connect it to a real API or allow it to search for information on the internet. This would enable the agent to verify a wider range of claims that are more up to date, rather than being limited to what we give it in advance.

---

## Final Reflection

### Trade-offs Between Reasoning Strategies

Complete the following table based on your experiments:

| Strategy             | Accuracy | Token Cost | Latency    | Best Use Case                                                                                  |
| -------------------- | -------- | ---------- | ---------- | ---------------------------------------------------------------------------------------------- |
| **Direct Prompting** | High     | Low       | Low       | Simple questions or when we just need a quick and inexpensive answer.                       |
| **Chain-of-Thought** | High     | High       | Med      | Problems that require several steps, logic, or more careful explanations.                   |
| **Tree-of-Thoughts** | Med    | Med/High | High       | Tasks where we want to explore several possible routes or when the solution is not so straightforward.    |
| **ReAct Agent**      | High     | Med      | Med/High | Situations where you need to verify data or combine reasoning with external tools. |


### Key Insights

**1. When does explicit reasoning (CoT) provide the most value?**

CoT is most useful when the problem needs a bit more calm, step-by-step thinking. When the answer isn't obvious, breaking things down helps you see the path more clearly instead of guessing or jumping too fast.

**2. What are the limitations of Tree-of-Thoughts compared to simpler approaches?**

Tree-of-Thoughts can get slow or feel heavier because it tries many possibilities before deciding. Sometimes that extra exploration isn't necessary, and a simpler method would reach the same answer without overthinking it.

**3. How does the ReAct pattern enable capabilities that pure reasoning cannot achieve?**

ReAct helps because it doesn't just "think”, it also takes action. Instead of staying stuck in the headspace of reasoning, it can look for information, try something, and then use that result to keep improving. Pure reasoning alone can't do that kind of back-and-forth.

**4. What connections do you see to the research assistant capstone project?**

All of this fits in with the final project because the assistant I am creating also needs to think, explore options, and take action when necessary. It's basically the same idea: combining reasoning with the ability to gather information and adapt on the fly so that the end result is more useful and reliable.